# ClickHouse

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/clickhouse.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is ClickHouse and how is it architected?

ClickHouse is a column-oriented OLAP database built for real-time analytics. It is shared-nothing: each node has local disks and its own CPU, and there is no single leader in the data path. A query can be sent to any node, and a `Distributed` table fans it out to shards.

+ **Table engine** - storage and behavior are chosen per table. `MergeTree` and its variants are the foundation; `Distributed` and `Replicated*` add scale and durability.
+ **Parts and partitions** - inserts create immutable parts; background merges combine them inside a partition.
+ **ClickHouse Keeper** - a Raft-based coordination service that replaces ZooKeeper for replication and distributed DDL.
+ **Clusters** - shards (horizontal data split) and replicas (copies of a shard), defined in `remote_servers` with per-node `macros`.

```text
config.d/remote_servers.xml
<clickhouse>
  <remote_servers>
    <analytics_cluster>
      <shard><replica><host>ch-1</host><port>9000</port></replica><replica><host>ch-2</host><port>9000</port></replica></shard>
      <shard><replica><host>ch-3</host><port>9000</port></replica><replica><host>ch-4</host><port>9000</port></replica></shard>
    </analytics_cluster>
  </remote_servers>
  <macros><shard>01</shard><replica>ch-1</replica></macros>
</clickhouse>
```

ClickHouse favors wide, append-only, denormalized tables with high insert throughput. It is not an OLTP database: there are no efficient single-row updates or foreign keys, and joins are a secondary operation rather than the primary access path.

A query arrives at any node. That node parses it, plans it, and splits the work into blocks of columns. Each block flows through a pipeline of operators, and the CPU works on vectors of values rather than one row at a time. This vectorized execution is why one node can scan billions of rows per second.

Data is stored on disk as one file per column per part. A query that names five columns touches only five column files, so a wide table stays cheap when queries are selective. Nothing is read for a column the query does not mention.

The server exposes its own state as tables under `system`. `system.parts` shows stored parts, `system.merges` shows running merges, `system.query_log` records finished queries, and `system.processes` shows what runs now.

```sql
SELECT database, table, sum(rows) AS rows, count() AS parts
FROM system.parts
WHERE active
GROUP BY database, table
ORDER BY parts DESC
LIMIT 5;
```

Common mistake: treating ClickHouse like an OLTP database. There is no cheap single-row update, no foreign keys, and joins are not the main access path. Model wide, append-only, denormalized tables, and reflect changes with `ReplacingMergeTree`.

## How does MergeTree store parts partitions and ORDER BY?

`MergeTree` sorts rows by the `ORDER BY` expression, splits data into partitions, writes each insert as a part, and merges parts in the background. The `PRIMARY KEY` is the sparse index prefix and, when omitted, defaults to `ORDER BY`.

```sql
CREATE TABLE analytics.events
(
    event_time  DateTime,
    user_id     UInt64,
    country     LowCardinality(String),
    event_type  LowCardinality(String),
    amount      Decimal(18, 2)
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(event_time)
ORDER BY (country, event_time, user_id)
SETTINGS index_granularity = 8192;

INSERT INTO analytics.events VALUES
    (now(), 101, 'DE', 'purchase', 42.50),
    (now(), 102, 'FR', 'view', 0);
```

Inspect the resulting layout with `SELECT partition, name, rows FROM system.parts WHERE table = 'events' AND active`.

Design rules:

+ `ORDER BY` should lead with the lowest-cardinality column you filter on, then the timestamp, then the high-cardinality key. Sorted data is what makes range scans fast.
+ Keep monthly or daily partitions for time-series data; too many tiny partitions hurt merges and metadata.
+ `PRIMARY KEY` must be a prefix of `ORDER BY`. Set it explicitly only when you want a shorter sparse index than the sort key.
+ A part is the unit of merging and each insert creates one part, so insert less often with more rows per insert.

An insert writes one or more immutable parts. A part covers exactly one partition and holds that insert's rows sorted by the `ORDER BY` key. Background merges combine several parts of the same partition into one larger part, and merges never cross partitions.

Every part is split into granules of `index_granularity` rows (8192 by default). Each granule gets one entry, a mark, in the primary index, and one mark per column file that points at the granule's byte offset. The index therefore holds roughly `rows / index_granularity` entries, which stays small enough to keep in memory.

Parts per partition is the read-side health metric. Each part means more files to open and more merge work to schedule. Aim for a few hundred active parts per partition at most. The server delays inserts when a partition passes `parts_to_delay_insert` (default 1000) and rejects them above `parts_to_throw_insert` (default 3000).

`ORDER BY` defines the physical sort order. `PRIMARY KEY` must be a prefix of it and defaults to the full `ORDER BY`. Set `PRIMARY KEY` shorter when you want a smaller index in memory but still want the longer sort key for range reads.

```sql
-- Long sort key for ordered scans, short index to keep memory low
CREATE TABLE analytics.events_pk
(
    event_date Date,
    country    LowCardinality(String),
    user_id    UInt64,
    event_time DateTime
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(event_date)
ORDER BY (country, event_date, user_id, event_time)
PRIMARY KEY (country, event_date);

-- Read parts and size per partition
SELECT partition, name, rows,
       formatReadableSize(bytes_on_disk) AS size,
       marks
FROM system.parts
WHERE table = 'events_pk' AND active
ORDER BY partition, name
LIMIT 20;
```

Read the row count of the result as the part count per partition. A partition with thousands of parts is a merge backlog, not a storage problem.

Common mistake: putting the most selective column first. A query that filters time alone cannot use an index that leads with `user_id`, so it reads everything. Lead with a column that appears in almost every `WHERE` clause.

In [ ]:
import math

ROWS = 5_000_000_000
INDEX_GRANULARITY = 8192

granules = math.ceil(ROWS / INDEX_GRANULARITY)
# One mark per granule in the primary index, plus one per granule in each key column file.
print(f"granules: {granules:,}")
print(f"marks if the index covers 3 columns: {granules * 3:,}")

assert granules == 610_352
assert granules * 3 == 1_831_056

## How do the sparse primary index and skipping indexes work?

The primary index is sparse: it stores one mark per granule of `index_granularity` rows (8192 by default) rather than one entry per row. A range scan finds the first matching granule and reads forward, which keeps the index small enough to stay in memory. Skipping indexes add secondary pruning for columns that are not in the sort key.

```sql
ALTER TABLE analytics.events
    ADD INDEX idx_type event_type TYPE set(100) GRANULARITY 4,
    ADD INDEX idx_amount amount TYPE minmax GRANULARITY 2,
    ADD INDEX idx_message message TYPE tokenbf_v1(8192, 3, 0) GRANULARITY 4;

ALTER TABLE analytics.events MATERIALIZE INDEX idx_type;

EXPLAIN indexes = 1
SELECT count() FROM analytics.events
WHERE event_type = 'purchase' AND amount > 1000;
```

The output reports which granules each index skipped:

```text
PrimaryKey  Keys: event_type, event_time, user_id   Granules: 116/1024
Skip        Name: idx_amount (minmax)               Granules: 32/116
```

Index guidance: `minmax` for numeric ranges, `set` for a bounded number of values, `bloom_filter` for equality on high-cardinality strings, and `tokenbf_v1` or `ngrambf_v1` for text search. Every index costs storage and merge time, so add one only after a query shows up in `system.query_log` as a full scan.

A mark maps a granule to a byte offset in a column file. The primary index stores marks only for the key columns, so an index lookup finds the first granule whose key can match, then reads forward until the key range ends.

Because the index is sparse, it can skip granules but not rows. A point lookup on the sort key still reads a whole granule of 8192 rows and filters inside it.

Skip indexes work on blocks of granules. `GRANULARITY n` groups n primary granules into one index block, so the index prunes at that coarser granularity. A smaller `GRANULARITY` prunes more precisely and costs more index entries.

```sql
-- Confirm what the primary index and each skip index skipped
EXPLAIN indexes = 1
SELECT count() FROM analytics.events
WHERE country = 'DE' AND event_time >= now() - INTERVAL 1 DAY;

-- Total marks available for the table, to compare against SelectedMarks
SELECT sum(marks) AS total_marks
FROM system.parts
WHERE database = 'analytics' AND table = 'events' AND active;

-- Marks actually read by recent queries
SELECT query, read_rows, ProfileEvents['SelectedMarks'] AS selected_marks
FROM system.query_log
WHERE type = 'QueryFinish' AND event_time > now() - INTERVAL 1 HOUR
ORDER BY read_rows DESC
LIMIT 5;
```

If `selected_marks` is close to `total_marks`, the sort key does not match the filter. Common mistake: adding a `minmax` index to a column queried with `LIKE '%text%'`; match the index type to the predicate instead.

## What are projections and when should you use them?

A projection is an alternative physical ordering of the same data, stored inside the table. It lets one table serve two access patterns without a second table or a materialized view.

```sql
ALTER TABLE analytics.events
    ADD PROJECTION by_user (
        SELECT user_id, event_time, event_type, amount
        ORDER BY (user_id, event_time)
    );

ALTER TABLE analytics.events MATERIALIZE PROJECTION by_user;

EXPLAIN indexes = 1
SELECT user_id, count() FROM analytics.events WHERE user_id = 101;
```

The planner selects a projection automatically when it fits the query. Use projections for a second common sort order or for pre-aggregated rollups on the same table. They cost extra disk and merge work, so keep the number small and materialize only what queries use. For rollups that must be shared and queried by name, a `MaterializedView` over `AggregatingMergeTree` is usually clearer.

A projection holds its own sort order and its own part files inside the same table. At insert and merge time the data is written once per projection. At read time the planner picks the projection whose sort order fits the `WHERE` and `GROUP BY`, and reads only that projection.

Projections are invisible to SQL. You query the table by name; the planner rewrites the plan. `system.projection_parts` reports their storage, and `system.projection_parts_columns` reports their columns.

```sql
ALTER TABLE analytics.events
    ADD PROJECTION by_user (
        SELECT user_id, event_time, event_type, amount
        ORDER BY (user_id, event_time)
    );
ALTER TABLE analytics.events MATERIALIZE PROJECTION by_user;

-- Confirm the planner chose the projection
EXPLAIN indexes = 1
SELECT user_id, count() FROM analytics.events WHERE user_id = 101 GROUP BY user_id;

-- Measure the extra storage each projection costs
SELECT name, sum(rows) AS rows,
       formatReadableSize(sum(bytes_on_disk)) AS size
FROM system.projection_parts
WHERE table = 'events' AND active
GROUP BY name;
```

Common mistake: adding a projection and skipping `MATERIALIZE PROJECTION`. New inserts fill it, but existing parts stay bare until it is materialized. Also keep the projection count small, because every insert and merge pays for each one.

## Which MergeTree engine variants exist?

The MergeTree family changes what happens during a merge. Choosing the right variant removes the need for application-side deduplication or rollup jobs.

| Engine | Merge behavior | Use case |
|---|---|---|
| `MergeTree` | keeps all rows | append-only facts and events |
| `ReplacingMergeTree` | keeps one row per sort key | upserts and CDC mirrors |
| `SummingMergeTree` | sums numeric columns per key | additive counters |
| `AggregatingMergeTree` | merges aggregate states | exact rollups with `uniq`, `avg`, `quantile` |
| `CollapsingMergeTree` | cancels rows by `sign` | streaming updates without versions |
| `VersionedCollapsingMergeTree` | cancels by `sign` and version | out-of-order updates |

```sql
CREATE TABLE analytics.orders_rt
(
    order_id   UInt64,
    status     LowCardinality(String),
    amount     Decimal(18, 2),
    updated_at DateTime
)
ENGINE = ReplacingMergeTree(updated_at)
ORDER BY order_id;

-- SummingMergeTree follows the same shape: ENGINE = SummingMergeTree((hits, bytes))
```

`ReplacingMergeTree` deduplicates only during merges, so duplicates remain visible until a merge runs; `FINAL` or an `argMax` aggregation is required for correctness. `SummingMergeTree` keeps one row per key after merging, so a pre-merge sum can differ from the post-merge value.

The engine variant decides what a merge does to rows that share the sort key.

`ReplacingMergeTree(version)` keeps the row with the largest version. Without a version column it keeps the last inserted row, which is only deterministic after a merge.

`SummingMergeTree(columns)` replaces a group of rows that share the sort key with one row whose listed numeric columns are summed. Only the listed columns are summed.

`AggregatingMergeTree` stores aggregate states in `AggregateFunction` columns and merges them with the matching `*Merge` function. It is the exact version of `SummingMergeTree` and supports `uniq`, `avg`, `quantile`, and `groupArray`.

```sql
CREATE TABLE analytics.page_sums
(
    day    Date,
    page   String,
    views  UInt64,
    errors UInt64
)
ENGINE = SummingMergeTree((views, errors))
ORDER BY (day, page);

INSERT INTO analytics.page_sums VALUES ('2026-10-04', '/pricing', 10, 1);
INSERT INTO analytics.page_sums VALUES ('2026-10-04', '/pricing', 5, 0);

-- Before a merge both rows are visible, so sum() is always correct
SELECT day, page, sum(views) AS views, sum(errors) AS errors
FROM analytics.page_sums
GROUP BY day, page;

-- Force the merge now
OPTIMIZE TABLE analytics.page_sums FINAL;
```

Common mistake: reading a `SummingMergeTree` without `sum()`. Duplicate keys stay visible until a merge runs, so a bare `SELECT views` can return several rows for one key.

## How do the Distributed and Replicated engines work?

`Replicated*MergeTree` keeps each shard's data on multiple nodes through ClickHouse Keeper. `Distributed` is a thin table that routes reads and writes across shards of a cluster.

```sql
CREATE TABLE analytics.events_local ON CLUSTER analytics_cluster
(
    event_time DateTime,
    user_id    UInt64,
    country    LowCardinality(String),
    amount     Decimal(18, 2)
)
ENGINE = ReplicatedMergeTree('/clickhouse/tables/{shard}/analytics/events', '{replica}')
PARTITION BY toYYYYMM(event_time)
ORDER BY (country, event_time);

CREATE TABLE analytics.events ON CLUSTER analytics_cluster AS analytics.events_local
ENGINE = Distributed('analytics_cluster', 'analytics', 'events_local', rand());

SELECT country, sum(amount) FROM analytics.events GROUP BY country;
```

Rules that avoid surprises: use `rand()` or a high-cardinality key for even distribution; use `GLOBAL JOIN` or `distributed_product_mode` for joins across `Distributed` tables; and write to a `Distributed` table only when you accept that the sharding key decides placement.

`ReplicatedMergeTree` coordinates replicas through ClickHouse Keeper. One replica takes the insert, writes the part locally, and records the part name in Keeper. Other replicas see the entry and fetch the part, so every replica eventually holds the same parts. If a replica falls behind, its `absolute_delay` grows and it catches up by copying parts from healthy replicas.

`Distributed` holds no data. It reads the shard list from `remote_servers`, sends the query to every shard, and merges the partial results on the initiator. For `sum(x)`, each shard returns a partial sum and the initiator adds them. For `GROUP BY`, each shard returns its groups and the initiator re-aggregates, so fan-out amplifies both latency and network traffic.

Shard design decides whether the fan-out is even. `rand()` spreads rows uniformly and makes every shard do equal work. A low-cardinality shard key creates a hot shard. A high-cardinality key such as `user_id` keeps data local for per-user queries but makes global aggregations read every shard.

```sql
-- Which shard holds which rows
SELECT hostName() AS node, count() AS rows
FROM analytics.events
GROUP BY node;

-- Replication health on every node
SELECT database, table, is_leader, is_readonly, absolute_delay, queue_size
FROM system.replicas
ORDER BY absolute_delay DESC;

-- Async fan-out write that waits for all shards
INSERT INTO analytics.events
SELECT * FROM analytics.events_stage
SETTINGS distributed_foreground_insert = 1;
```

Common mistake: joining two `Distributed` tables without `GLOBAL`. Without it, each shard builds a hash table from its own slice of the right table, so a left row on one shard cannot match a right row that lives on another. Use `GLOBAL JOIN`, or set `distributed_product_mode = 'global'`.

## How does TTL manage the data lifecycle?

TTL statements run during merges to delete, move, or aggregate old data. Retention becomes part of the table definition instead of a batch job.

```sql
CREATE TABLE analytics.logs
(
    log_time DateTime,
    service  LowCardinality(String),
    level    LowCardinality(String),
    message  String
)
ENGINE = MergeTree()
PARTITION BY toDate(log_time)
ORDER BY (service, log_time)
TTL log_time + INTERVAL 30 DAY DELETE,
    log_time + INTERVAL 7 DAY TO VOLUME 'cold';

-- Aggregating TTL: keep hourly error rollups longer than raw rows
ALTER TABLE analytics.logs
    MODIFY TTL
        log_time + INTERVAL 7 DAY DELETE,
        log_time + INTERVAL 7 DAY GROUP BY toStartOfHour(log_time), service, level
            SET message = '', hits = count()
        WHERE level = 'error';
```

TTL is applied on merge, not on a wall clock, so a partition can briefly outlive its TTL. Pair `DELETE` TTLs with partition-level `DROP PARTITION` for hard guarantees, and use `TO VOLUME` or `TO DISK` with a storage policy that defines hot and cold tiers.

TTL rules are stored in the table metadata and evaluated while parts merge. A `DELETE` rule drops rows whose expression is in the past. A `TO VOLUME` or `TO DISK` rule moves a whole part to another storage tier once every row in the part passes the expression. A `GROUP BY` rule rewrites rows into aggregate states after a delay.

The TTL merge runs on its own schedule, controlled by the MergeTree setting `merge_with_ttl_timeout`, separate from normal merges. Because `TO VOLUME` moves a part only when all of its rows satisfy the rule, a part that spans a wide time range delays the move.

```sql
ALTER TABLE analytics.logs
    MODIFY TTL
        log_time + INTERVAL 30 DAY DELETE,
        log_time + INTERVAL 7 DAY TO VOLUME 'cold';

-- Where parts currently sit
SELECT disk_name, count() AS parts, sum(rows) AS rows
FROM system.parts
WHERE table = 'logs' AND active
GROUP BY disk_name;

-- Hard retention: drop a whole partition instead of waiting for TTL
ALTER TABLE analytics.logs DROP PARTITION '20260101';
```

Common mistake: expecting TTL to delete on a wall clock. A row survives until a merge that touches its part runs, so a low-insert table can keep expired data far past the TTL. Drop old partitions on a schedule when retention must be exact, and use TTL as the steady-state mechanism.

## How do materialized views do incremental aggregation?

A ClickHouse materialized view is a trigger on inserts, not a cached query result. It reads the inserted block and writes to a target table, which makes it the standard tool for incremental aggregation.

```sql
CREATE TABLE analytics.daily_country
(
    day     Date,
    country LowCardinality(String),
    users   AggregateFunction(uniq, UInt64),
    revenue SimpleAggregateFunction(sum, Decimal(18, 2)),
    events  SimpleAggregateFunction(sum, UInt64)
)
ENGINE = AggregatingMergeTree()
ORDER BY (day, country);

CREATE MATERIALIZED VIEW analytics.mv_daily_country TO analytics.daily_country AS
SELECT
    toDate(event_time) AS day,
    country,
    uniqState(user_id) AS users,
    sum(amount)        AS revenue,
    count()            AS events
FROM analytics.events
GROUP BY day, country;
```

Only rows inserted after the view is created are processed, so create the target table and view before backfilling raw data, or backfill the target explicitly. Use `AggregateFunction` for exact distinct or quantiles and `SimpleAggregateFunction` for plain `sum`, `min`, `max`, and `any`.

A materialized view is a stored query with a trigger. On every insert into the source table, ClickHouse runs the view's `SELECT` over just the inserted block and writes the result to the target table. It never reads historical data. The `TO target` form makes the view a thin trigger; without it, the view creates and owns its own target table.

Because the view only ever sees inserted blocks, correctness depends on the `GROUP BY` producing partial states. `AggregatingMergeTree` stores those states and merges them later, so the rollup stays exact across many inserts. A plain `SummingMergeTree` or `ReplacingMergeTree` works the same way for additive counters and last-write-wins state.

```sql
-- Confirm the view is firing on inserts
SELECT event_time, query_duration_ms, read_rows, written_rows
FROM system.query_log
WHERE query LIKE '%mv_daily_country%' AND type = 'QueryFinish'
ORDER BY event_time DESC
LIMIT 5;

-- Compare target size with source size
SELECT table, count() AS parts, sum(rows) AS rows
FROM system.parts
WHERE table IN ('daily_country', 'events') AND active
GROUP BY table;
```

Common mistake: creating the materialized view after backfilling the source table. Rows already in the source never pass through the view, so the target stays empty for them. Create the target and view first, then load, or backfill the target explicitly with `INSERT INTO target SELECT ... GROUP BY`.

## How do you ingest from Kafka with the Kafka engine?

The Kafka table engine turns a topic into a table. Reads consume messages, and a materialized view on top of it is the usual way to land data into a `MergeTree` target.

```sql
CREATE TABLE analytics.events_kafka
(
    event_time DateTime,
    user_id    UInt64,
    country    LowCardinality(String),
    amount     Decimal(18, 2)
)
ENGINE = Kafka
SETTINGS
    kafka_broker_list = 'kafka-1:9092,kafka-2:9092',
    kafka_topic_list = 'events',
    kafka_group_name = 'clickhouse-events',
    kafka_format = 'JSONEachRow',
    kafka_num_consumers = 4,
    kafka_max_block_size = 65536,
    kafka_skip_broken_messages = 100;

CREATE MATERIALIZED VIEW analytics.mv_events_kafka TO analytics.events AS
SELECT event_time, user_id, country, amount
FROM analytics.events_kafka;
```

The Kafka engine table is a cursor, not storage, so never query it for analytics directly. Duplicate consumption is normal, so make the target idempotent or use `ReplacingMergeTree`. A parse error blocks the consumer unless `kafka_skip_broken_messages` is set.

The Kafka engine owns consumer groups and stores offsets in Kafka, not in ClickHouse. Reading the table starts the consumers; a `SELECT` returns the messages read so far and advances the offset. A materialized view makes the read continuous, because the view's insert trigger keeps pulling blocks.

Each `kafka_num_consumers` adds one consumer to the group and one parsing thread. `kafka_max_block_size` controls how many messages form one insert block, so keep it in the thousands to avoid tiny parts. `kafka_skip_broken_messages` lets the consumer survive malformed records instead of stopping.

```sql
-- Kafka consumption counters owned by this server
SELECT name, value FROM system.events
WHERE name LIKE 'KafkaMessages%';

-- Restart consumption after changing the table settings
DETACH TABLE analytics.events_kafka;
ATTACH TABLE analytics.events_kafka;
```

Common mistake: adding consumers to fix lag when the target table is the bottleneck. More consumers mean more concurrent small inserts and more parts. Raise `kafka_max_block_size` and check `system.merges` before scaling consumers.

## How do async inserts batch inserts and Kafka Connect differ?

ClickHouse is built for bulk inserts. Row-by-row `INSERT ... VALUES` is the classic anti-pattern, because every insert creates a part and many small parts trigger merge pressure.

Async inserts let the server buffer small writes and form larger blocks, which suits many clients that each send a row at a time. Batch inserts are better when the client controls the flow.

```sql
INSERT INTO analytics.events SETTINGS
    async_insert = 1,
    wait_for_async_insert = 1,
    async_insert_max_data_size = 10485760,
    async_insert_busy_timeout_ms = 1000
VALUES (now(), 101, 'DE', 'purchase', 42.50);
```

In [ ]:
# ILLUSTRATIVE - requires clickhouse_connect: install to run, or read on.
import clickhouse_connect

client = clickhouse_connect.get_client(host="ch-1", port=8123, database="analytics")

rows = [
    ("2026-10-04 10:00:00", 101, "DE", "purchase", 42.50),
    ("2026-10-04 10:00:01", 102, "FR", "view", 0.00),
]

client.insert(
    "events",
    rows,
    column_names=["event_time", "user_id", "country", "event_type", "amount"],
    settings={"async_insert": 1, "wait_for_async_insert": 1, "max_insert_threads": 4},
)

`max_insert_block_size` (default 1048576 rows) caps the block the server forms, `min_insert_block_size_rows` and `min_insert_block_size_bytes` control squashing before parts are written, `input_format_parallel_parsing` speeds up large file loads, and `async_insert_deduplicate` deduplicates identical buffered inserts on retries. Aim for at least 10k to 100k rows per part on a busy table.

The ClickHouse Kafka Connect sink is the alternative when you already run Kafka Connect and want exactly-once delivery and connector management in one place.

```json
{
  "name": "clickhouse-events-sink",
  "config": {
    "connector.class": "com.clickhouse.kafka.connect.ClickHouseSinkConnector",
    "tasks.max": "4",
    "topics": "events",
    "hostname": "ch-1",
    "port": "8443",
    "database": "analytics",
    "username": "ingest",
    "ssl": "true",
    "exactlyOnce": "true",
    "topic2TableMap": "events=events_raw"
  }
}
```

```bash
curl -s -X POST -H 'Content-Type: application/json' \
  --data @clickhouse-sink.json http://connect:8083/connectors
```

The Kafka engine is lower-latency and lives inside ClickHouse, but offsets and parse errors are managed by the server. Kafka Connect gives exactly-once semantics and DLQ handling at the cost of another service to operate. Both write into the same `MergeTree` targets.

An insert creates at least one part per partition it touches. A client that inserts one row per statement creates one part per row, and merges cannot keep up with that rate. Async inserts solve this on the server: rows wait in a buffer until `async_insert_busy_timeout_ms` expires or `async_insert_max_data_size` is reached, then the server writes one part.

Batch inserts solve it on the client: buffer rows and send 10k to 100k per statement. This is the fastest path and the one to prefer whenever the client controls the cadence. `min_insert_block_size_rows` and `min_insert_block_size_bytes` also squash blocks before they become parts.

```sql
-- Parts and rows per table right now
SELECT table, count() AS parts, sum(rows) AS rows, max(rows) AS biggest_part
FROM system.parts
WHERE active AND database = 'analytics'
GROUP BY table
ORDER BY parts DESC;

-- The thresholds that first delay, then reject inserts
SELECT name, value, description
FROM system.merge_tree_settings
WHERE name IN ('parts_to_delay_insert', 'parts_to_throw_insert', 'max_parts_in_total');

-- Merge backlog
SELECT database, table, round(elapsed) AS elapsed_s, num_parts
FROM system.merges
ORDER BY elapsed DESC;
```

Common mistake: leaving `async_insert = 1` on for controlled bulk loads. It adds latency and hides a client that should batch. Turn it on for many small writers; turn it off for bulk loads.

In [ ]:
# ILLUSTRATIVE - requires clickhouse_connect: install to run, or read on.
# Review only: the clickhouse-connect driver is not installed in this repository.
import clickhouse_connect

client = clickhouse_connect.get_client(host="ch-1", port=8123, database="analytics")

# Ask the server how many parts each table has before deciding to batch more.
parts = client.query(
    "SELECT table, count() AS parts, sum(rows) AS rows "
    "FROM system.parts WHERE active AND database = 'analytics' "
    "GROUP BY table ORDER BY parts DESC"
)
for table, part_count, rows in parts.result_rows:
    print(f"{table}: {part_count} parts, {rows:,} rows")

## How do codecs and compression work?

ClickHouse compresses every column independently and lets you choose a codec per column. Codecs exploit the column's value pattern, which is why the columnar layout compresses far better than row storage.

```sql
CREATE TABLE analytics.metrics
(
    ts          DateTime CODEC(DoubleDelta, ZSTD(3)),
    device_id   UInt64 CODEC(Delta, ZSTD(3)),
    temperature Float32 CODEC(Gorilla, ZSTD(3)),
    status      LowCardinality(String) CODEC(ZSTD(3)),
    raw         String CODEC(NONE)
)
ENGINE = MergeTree()
PARTITION BY toDate(ts)
ORDER BY (device_id, ts);

SELECT name, formatReadableSize(sum(data_uncompressed_bytes) / sum(data_compressed_bytes)) AS ratio
FROM system.columns WHERE table = 'metrics' GROUP BY name;
```

`Delta` and `DoubleDelta` are excellent for monotonically increasing timestamps and counters, `Gorilla` for slowly changing floats, `T64` for small integer ranges, and `ZSTD` as the general-purpose backstop. Higher ZSTD levels save storage but cost insert CPU. Check the real ratio in `system.columns` rather than assuming, and leave already-compressed payloads as `NONE`.

Each column is compressed on its own with a chain of codecs applied left to right. The first codec transforms the values, the last compresses bytes. `CODEC(Delta, ZSTD(3))` delta-encodes the column and then compresses the deltas with ZSTD.

The codec that wins depends on the value pattern. Monotonic timestamps and counters compress well after delta. Small integer ranges fit `T64`. Slowly changing floats fit `Gorilla`. General text fits `ZSTD`. Compression runs at insert and merge time, so a higher level trades insert CPU for disk space.

```sql
SELECT
    name,
    formatReadableSize(sum(data_compressed_bytes))   AS compressed,
    formatReadableSize(sum(data_uncompressed_bytes)) AS uncompressed,
    round(sum(data_uncompressed_bytes) / sum(data_compressed_bytes), 2) AS ratio
FROM system.columns
WHERE database = 'analytics' AND table = 'metrics'
GROUP BY name
ORDER BY sum(data_uncompressed_bytes) DESC;
```

Common mistake: applying `ZSTD(19)` to every column. The insert and merge path pays that CPU on every part, and merges are usually the write bottleneck. Default to level 1 to 3 and reserve high levels for cold data. Leave already-compressed payloads, such as images or gzip blobs, as `CODEC(NONE)`.

## Which column types matter and what are the Nullable caveats?

Type choice drives both compression and query speed. The biggest wins come from replacing generic strings with specialized types.

```sql
CREATE TABLE analytics.sessions
(
    session_id  UUID,
    user_id     UInt64,
    country     LowCardinality(String),
    device      Enum8('desktop' = 1, 'mobile' = 2, 'tablet' = 3),
    started_at  DateTime64(3, 'UTC'),
    duration_ms UInt32,
    revenue     Decimal(18, 2),
    tags        Array(LowCardinality(String)),
    props       Map(String, String)
)
ENGINE = MergeTree()
ORDER BY (country, started_at);
```

+ `LowCardinality(String)` dictionary-encodes a column with fewer than roughly 10k distinct values. It saves space and speeds up grouping and filtering, but hurts if cardinality is actually high.
+ `Enum8` or `Enum16` is cheaper than `LowCardinality(String)` when the value set is fixed and known.
+ `Array`, `Map`, and `Tuple` are first-class, and array functions avoid the row explosion that join-then-group would cause.

`Nullable(T)` stores a separate null mask, blocks some optimizations, and cannot be used efficiently in the sort key, so prefer a default or an explicit sentinel.

```sql
-- Prefer a plain column with a default over Nullable on the hot path
CREATE TABLE analytics.events_no_null
(event_time DateTime, country LowCardinality(String) DEFAULT 'unknown')
ENGINE = MergeTree() ORDER BY (country, event_time);
```

Avoid `Nullable` inside `AggregateFunction` states and in join keys; `LowCardinality(Nullable(String))` is supported but compounds the cost. Keep one or two `Nullable` columns at the edges of a table, not across the sort key.

`LowCardinality(String)` stores each distinct value once in a dictionary and keeps a small integer key per row. Filtering and grouping then compare integers, which is fast and compact. The win disappears as the number of distinct values approaches the row count, because the dictionary grows and every read pays a lookup. Measure `uniqExact` before you choose it.

`Enum8` and `Enum16` store a fixed value set as a one-byte or two-byte code with named labels, which is cheaper than `LowCardinality` when the set is known and does not change. `Array`, `Map`, and `Tuple` are first-class, and array functions avoid the row explosion a join-then-group would cause.

```sql
-- Check real cardinality before choosing LowCardinality
SELECT uniqExact(country) AS country_values,
       uniqExact(user_id) AS user_values,
       count()            AS rows
FROM analytics.events;

-- Measure the storage of the columns you are comparing
SELECT name, formatReadableSize(sum(data_compressed_bytes)) AS size
FROM system.columns
WHERE table = 'events' AND name IN ('country', 'optional_field')
GROUP BY name;
```

Common mistake: wrapping every column in `Nullable` "just in case". `Nullable` adds a null mask, blocks some index and aggregation optimizations, and is awkward in a sort key. Prefer a plain column with a `DEFAULT`, and use `Nullable` only where NULL is a real business state.

## How do ClickHouse JOIN algorithms work?

Joins in ClickHouse are hash-based by default and the build side is the right-hand table. Distribution, memory, and algorithm all matter more than in a row-store.

```sql
SET join_algorithm = 'hash';
SET max_bytes_in_join = 20000000000;
SET join_use_nulls = 1;

SELECT e.country, d.region, count()
FROM analytics.events AS e
LEFT JOIN analytics.dict_country AS d ON e.country = d.country
GROUP BY e.country, d.region;
```

+ `hash` (default) - builds a hash table from the right side. Fast when the right side fits in memory.
+ `grace_hash` - hash join that spills to disk; the safest choice for large joins.
+ `parallel_hash` - parallel build and probe, good for large right sides on many cores.
+ `partial_merge` and `full_sorting_merge` - sort-merge joins that avoid a full hash table.
+ `direct` - uses the right table's storage order directly, with no hash table.
+ `GLOBAL JOIN` - required when joining two `Distributed` tables so the right side is sent to each shard once.

Keep the right-hand table small, watch `max_bytes_in_join` and `join_algorithm` in `system.query_log`, and prefer dictionaries over joins for repeated lookups.

ClickHouse reads the right table first and builds a hash table from its join key, unless the algorithm says otherwise. The right table is the build side and the left table streams through the probe. Memory is the limit, so the right table must be the small one.

`hash` keeps the whole right side in memory. `grace_hash` partitions both sides and spills partitions to disk. `parallel_hash` builds the table with several threads. `partial_merge` and `full_sorting_merge` sort both sides and merge. `direct` reads the right table in key order with no hash table, which works when the right table is small and has no duplicate keys.

```sql
-- Force a spilling join and cap its memory
SET join_algorithm = 'grace_hash';
SET max_bytes_in_join = 20000000000;

SELECT e.country, c.region, count() AS events
FROM analytics.events AS e
LEFT JOIN analytics.country_ref AS c ON e.country = c.country
GROUP BY e.country, c.region
ORDER BY events DESC;

-- Inspect the join in the plan
EXPLAIN actions = 1
SELECT count() FROM analytics.events AS e
LEFT JOIN analytics.country_ref AS c ON e.country = c.country;
```

Common mistake: putting the large table on the right. The build side consumes `max_bytes_in_join`, and exceeding it fails the query with a memory-limit error unless the algorithm spills. Swap the join so the small table is on the right, or replace the lookup with `dictGet`.

## How do dictionaries speed up lookups?

Dictionaries are key-value lookups loaded into memory or partly on SSD and refreshed on a lifetime. They replace a repeated join with a function call.

```sql
CREATE DICTIONARY analytics.country_region
(
    country String,
    region  String,
    manager String
)
PRIMARY KEY country
SOURCE(CLICKHOUSE(host 'ch-1' port 9000 db 'analytics' table 'country_reference' user 'default'))
LAYOUT(HASHED())
LIFETIME(MIN 300 MAX 600);

SELECT e.country,
       dictGet('analytics.country_region', 'region', e.country) AS region,
       count() AS events
FROM analytics.events AS e
WHERE e.event_time >= today() - 1
GROUP BY e.country, region
ORDER BY events DESC;
```

Layouts: `FLAT` for small numeric keys, `HASHED` for general keys, `COMPLEX_KEY_HASHED` for composite keys, `RANGE_HASHED` for time-bounded attributes, and `CACHE` or `SSD_CACHE` when the reference data is too large for memory. An unloaded or stale dictionary shows up in `system.dictionaries` as `status = 'NOT_LOADED'` or with an old `last_successful_update_time`, so monitor it like a job.

A dictionary loads its source into a memory layout and answers `dictGet` calls with a hash lookup. The layout decides the cost. `FLAT` is an array indexed by key and needs small numeric keys. `HASHED` is a hash table for any key. `COMPLEX_KEY_HASHED` handles a tuple key. `RANGE_HASHED` returns the attribute whose range contains the key. `CACHE` and `SSD_CACHE` keep only the hot keys.

`LIFETIME` controls refresh. `LIFETIME(MIN 300 MAX 600)` reloads at a random time between 300 and 600 seconds, which spreads the reload cost across dictionaries instead of syncing them on the same tick.

```sql
-- Dictionary health, size, and staleness
SELECT name,
       status,
       element_count,
       formatReadableSize(bytes_allocated) AS memory,
       last_successful_update_time,
       last_exception
FROM system.dictionaries
ORDER BY bytes_allocated DESC;
```

Common mistake: leaving a stale dictionary unnoticed. If the source query fails, the dictionary keeps the last good copy and queries silently return old values. Alert on `last_exception` and on `last_successful_update_time` growing older than the configured `LIFETIME` maximum.

## How do you read EXPLAIN output and optimize queries?

`EXPLAIN` has several modes, and `indexes = 1` is the one that tells you whether the query prunes data or scans everything.

```sql
EXPLAIN PIPELINE SELECT country, count() FROM analytics.events GROUP BY country;
EXPLAIN indexes = 1 SELECT count() FROM analytics.events WHERE country = 'DE';
```

Then confirm with the query log:

```sql
SELECT query_duration_ms,
       read_rows,
       read_bytes,
       memory_usage,
       ProfileEvents['SelectedMarks'] AS marks
FROM system.query_log
WHERE type = 'QueryFinish' AND query LIKE '%analytics.events%'
ORDER BY event_time DESC
LIMIT 10;
```

Optimization checklist:

+ Filter on the leading `ORDER BY` column so `PrimaryKey` granules are skipped.
+ Add a skipping index only for a column that appears in slow queries.
+ Reduce `SelectedMarks`; if it equals total marks, the sort key does not match the filter.
+ Replace `SELECT *` with explicit columns, because wide tables read many column files.
+ Prefer `uniqCombined` for approximate distinct counts and `AggregatingMergeTree` for repeated rollups.
+ Use `max_threads`, `max_memory_usage`, and `max_execution_time` as per-query guardrails, tracked in `system.processes`.

`EXPLAIN indexes = 1` shows the primary key ranges and skip indexes the planner will use, with granule counts before and after each index. `EXPLAIN PIPELINE` shows the operators, their thread counts, and whether the plan is parallel. `EXPLAIN actions = 1` expands the query into the functions the engine actually calls.

`system.query_log` is the ground truth for what a query cost. `read_rows` and `read_bytes` show scan volume, `memory_usage` the peak, `ProfileEvents['SelectedMarks']` the index work, and `result_rows` the output size. For a heavy aggregation, `max_bytes_before_external_group_by` spills the aggregation state to disk instead of failing on the memory limit, at the cost of disk I/O.

```sql
-- Slowest queries in the last day, by bytes read
SELECT
    normalized_query_hash,
    any(query)                               AS sample,
    count()                                  AS runs,
    round(avg(query_duration_ms))            AS avg_ms,
    round(quantile(0.95)(query_duration_ms)) AS p95_ms,
    sum(read_rows)                           AS rows_read,
    formatReadableSize(sum(read_bytes))      AS bytes_read
FROM system.query_log
WHERE type = 'QueryFinish'
  AND event_time > now() - INTERVAL 1 DAY
  AND query_kind = 'Select'
GROUP BY normalized_query_hash
ORDER BY sum(read_bytes) DESC
LIMIT 10;
```

Common mistake: tuning by wall-clock time alone. A query that finishes in 50 ms but reads a billion rows looks fine on a small table and collapses at 100x the data. Watch `read_bytes` and `SelectedMarks` to see whether the work is proportional to the answer or to the whole table.

In [ ]:
# ILLUSTRATIVE - requires clickhouse_connect: install to run, or read on.
# Review only: clickhouse-connect is not installed in this repository.
import clickhouse_connect

client = clickhouse_connect.get_client(host="ch-1", port=8123, database="analytics")

result = client.query(
    "SELECT normalized_query_hash, any(query) AS sample, count() AS runs, "
    "       round(avg(query_duration_ms)) AS avg_ms, sum(read_bytes) AS bytes_read "
    "FROM system.query_log "
    "WHERE type = 'QueryFinish' AND event_time > now() - INTERVAL 1 DAY "
    "GROUP BY normalized_query_hash ORDER BY bytes_read DESC LIMIT 10"
)
for row in result.result_rows:
    print(row)

## How do you deduplicate with ReplacingMergeTree and FINAL?

`ReplacingMergeTree` keeps the row with the greatest version per sort key during a merge. Because merges are asynchronous, duplicates remain queryable until then, and `FINAL` resolves them at read time.

```sql
CREATE TABLE analytics.users_rt
(
    user_id    UInt64,
    plan       LowCardinality(String),
    email      String,
    updated_at DateTime,
    is_deleted UInt8 DEFAULT 0
)
ENGINE = ReplacingMergeTree(updated_at, is_deleted)
ORDER BY user_id;
```

Three ways to read the deduplicated state:

```sql
-- Correct but merges everything at query time
SELECT * FROM analytics.users_rt FINAL WHERE is_deleted = 0;

-- Faster on large tables: aggregate at query time
SELECT user_id,
       argMax(plan, updated_at) AS plan,
       argMax(email, updated_at) AS email
FROM analytics.users_rt
GROUP BY user_id
HAVING argMax(is_deleted, updated_at) = 0;

-- Force a merge now, then query without FINAL
OPTIMIZE TABLE analytics.users_rt FINAL;
```

`FINAL` cost is controlled by `max_final_threads` and `do_not_merge_across_partitions_select_final`. For a steady-state production table, prefer the `argMax` pattern or partition by a key so `FINAL` only merges within one partition.

`FINAL` resolves duplicates at read time by merging the selected parts on the fly. The cost is controlled by `max_final_threads` and `do_not_merge_across_partitions_select_final`, which lets `FINAL` work partition by partition instead of across all of them. Partitioning the table by a coarse time key makes that optimization effective.

The `argMax` pattern avoids `FINAL` entirely. Group by the sort key and pick the value of each other column at the greatest version. It reads more rows through aggregation but stays predictable as the table grows, and it composes with filters.

```sql
-- Version and deletion state per key, without FINAL
SELECT user_id,
       argMax(plan, updated_at)       AS plan,
       argMax(email, updated_at)      AS email,
       max(updated_at)                AS last_update
FROM analytics.users_rt
GROUP BY user_id
HAVING argMax(is_deleted, updated_at) = 0
ORDER BY last_update DESC
LIMIT 20;

-- See how many duplicate versions exist before a merge
SELECT count() AS rows, uniqExact(user_id) AS keys,
       count() / uniqExact(user_id) AS versions_per_key
FROM analytics.users_rt;
```

Common mistake: running `OPTIMIZE ... FINAL` on a large table in a loop to keep it deduplicated. `OPTIMIZE` rewrites parts and competes with normal merges for I/O. Schedule it rarely, or rely on `argMax` at read time.

## How do you build real-time analytics pipelines?

A real-time pipeline on ClickHouse is usually a raw table, one or more incremental materialized views, and a query layer over aggregated tables.

```sql
-- Raw events land in a short-lived MergeTree with TTL (see the TTL section)
CREATE TABLE analytics.events_minute
(
    minute  DateTime,
    country LowCardinality(String),
    events  SimpleAggregateFunction(sum, UInt64),
    revenue SimpleAggregateFunction(sum, Decimal(18, 2)),
    users   AggregateFunction(uniq, UInt64)
)
ENGINE = AggregatingMergeTree()
PARTITION BY toDate(minute)
ORDER BY (country, minute);

CREATE MATERIALIZED VIEW analytics.mv_events_minute TO analytics.events_minute AS
SELECT
    toStartOfMinute(event_time) AS minute,
    country,
    count()            AS events,
    sum(amount)        AS revenue,
    uniqState(user_id) AS users
FROM analytics.events_raw
GROUP BY minute, country;

SELECT country, sum(events) AS events, sum(revenue) AS revenue, uniqMerge(users) AS users
FROM analytics.events_minute
WHERE minute >= now() - INTERVAL '60' MINUTE
GROUP BY country
ORDER BY revenue DESC;
```

Patterns worth copying: keep raw data short-lived and rollups long-lived; make every stage idempotent because Kafka ingestion is at-least-once; and use `ReplacingMergeTree` for dimensions that change, with a periodic `OPTIMIZE` for readers that cannot tolerate duplicates.

Every stage of the pipeline must be idempotent because Kafka delivery is at-least-once. A raw table keyed by `(event_id)` in a `ReplacingMergeTree`, or a deduplicating insert block, absorbs replays. Rollup tables fed by `AggregatingMergeTree` states are naturally idempotent per insert block only if the source block is not duplicated, so deduplicate upstream or key the raw table.

Operate the pipeline by watching the stages. `system.parts` shows whether a rollup is growing, `system.merges` shows whether merges keep up, and `system.query_log` shows whether the views fire on every insert.

```sql
-- Raw events are deduplicated on replay by event_id
CREATE TABLE analytics.events_raw
(
    event_id   UUID,
    event_time DateTime,
    country    LowCardinality(String),
    amount     Decimal(18, 2)
)
ENGINE = ReplacingMergeTree()
PARTITION BY toDate(event_time)
ORDER BY event_id
TTL event_time + INTERVAL 3 DAY DELETE;

-- Confirm each stage is alive
SELECT table, count() AS parts, sum(rows) AS rows
FROM system.parts
WHERE database = 'analytics' AND active
  AND table IN ('events_raw', 'events_minute', 'daily_country')
GROUP BY table
ORDER BY parts DESC;
```

Common mistake: building the rollup from the raw table with `SELECT ... FROM events_raw FINAL` inside a scheduled job. That scans and merges raw data on every run. Let the materialized view update the rollup on insert, and reserve the scheduled job for a periodic repair or backfill.

## How does ClickHouse compare with Druid BigQuery and Snowflake?

| System | Storage and compute | Strength | Trade-off |
|---|---|---|---|
| ClickHouse | tightly coupled per node, shared-nothing | sub-second queries on high-volume event data, self-managed or cloud | operational effort, limited multi-statement transactions |
| Druid | separated, segment-based | pre-aggregated time-series with rollup at ingestion | complex ingestion specs, weaker general SQL |
| BigQuery | fully separated, serverless | elastic scale, zero operations, big ecosystem | cost per query, less control over physical layout |
| Snowflake | separated, virtual warehouses | concurrency isolation, mature governance, SQL breadth | cost, higher latency for high-frequency small queries |

```text
Choose ClickHouse for high-volume append-only events with sub-second aggregation
targets and a team that can operate a cluster; choose BigQuery or Snowflake when
operations, governance, and elasticity matter more than hardware-level control.
```

ClickHouse also handles workloads the separated systems do not: `clickhouse-local` for ad-hoc files, S3 and Iceberg table functions for querying a lake in place, and materialized views that keep rollups fresher than a scheduled warehouse job.

ClickHouse's closest cousin is Druid: both pre-aggregate at ingestion and serve time-series rollups. ClickHouse is closer to a general SQL database, with full SQL, joins, dictionaries, and table engines that read files directly. BigQuery and Snowflake separate storage from compute, which makes them elastic and low-maintenance. ClickHouse couples them, which makes each node fast but makes resizing a rebalancing exercise rather than a slider. Choose on the operating model and on latency targets, not only on benchmark numbers.

## What operational concerns matter in production?

ClickHouse rewards proactive operations. Three areas cause most incidents.

**Merges.** Merge throughput must keep up with insert rate, or parts accumulate and reads slow down.

```sql
SELECT database, table, elapsed, progress, num_parts FROM system.merges ORDER BY elapsed DESC;
```

**Keeper and replication.** Replication correctness depends on ClickHouse Keeper being healthy and quorum-connected.

```sql
SELECT * FROM system.replicas WHERE is_readonly OR absolute_delay > 60;
SELECT * FROM system.replication_queue ORDER BY create_time LIMIT 20;
SELECT * FROM system.zookeeper WHERE path = '/clickhouse/tables';
```

**Mutations, backups, and guardrails.** `ALTER TABLE ... UPDATE` and `DELETE` are asynchronous rewrites, not point updates.

```sql
ALTER TABLE analytics.events DELETE WHERE event_time < today() - 365;
SELECT * FROM system.mutations WHERE is_done = 0;
BACKUP TABLE analytics.events TO Disk('backups', 'events.zip');
```

Configure storage policies for hot and cold volumes, cap concurrency and memory with `max_concurrent_queries`, `max_memory_usage`, and `max_threads`, and monitor `system.errors`, `system.events`, and disk free space, because `MergeTree` stops accepting inserts when a disk fills. Run DDL with `ON CLUSTER` so every node stays in sync.

**Merge backpressure.** When inserts outrun merges, parts accumulate, reads slow down, and the server first delays then rejects inserts. Watch the parts-per-partition distribution, not just the total, because one hot partition can stall a table.

```sql
SELECT table, partition, count() AS parts, sum(rows) AS rows
FROM system.parts
WHERE active AND database = 'analytics'
GROUP BY table, partition
ORDER BY parts DESC
LIMIT 10;

SELECT database, table, round(elapsed) AS elapsed_s, num_parts
FROM system.merges
ORDER BY elapsed DESC;
```

The fix is upstream: batch inserts, lower the insert rate, or add shards. Raising `parts_to_delay_insert` only hides the backlog.

**Keeper and replication health.** Replication correctness depends on ClickHouse Keeper having quorum. A replica that loses Keeper goes read-only, and `absolute_delay` shows how far behind it is. Check `system.replicas`, the replication queue, and Keeper connectivity together before restarting anything.

```sql
SELECT database, table, is_readonly, absolute_delay, queue_size
FROM system.replicas
WHERE is_readonly OR absolute_delay > 60;

SELECT * FROM system.replication_queue ORDER BY create_time LIMIT 20;
```

**Mutations and guardrails.** `ALTER TABLE ... UPDATE` and `DELETE` are asynchronous part rewrites, not point updates; watch `system.mutations` for progress and stuck mutations. Cap resource use with `max_concurrent_queries`, `max_memory_usage`, `max_threads`, and `max_bytes_before_external_group_by`, and monitor disk free space, because `MergeTree` stops accepting inserts when a disk fills.

## How do you design the ORDER BY and primary key from real filters?

Start from the queries, not from the schema. List the columns each query filters on, and rank them by how often they appear. The leading `ORDER BY` column must be one that almost every query constrains, because the sparse index can only skip when a filter restricts a prefix of the key.

Order the key as: equality filters first, then the time column, then high-cardinality columns used for ordering. A filter on a prefix of `ORDER BY` prunes granules. A filter on a later column alone does not.

```sql
-- Query patterns to design for:
--   1. WHERE country = ? AND event_time >= ?   -> (country, event_time)
--   2. WHERE user_id = ?                       -> projection on (user_id)
--   3. WHERE event_type = ?                    -> skip index or nothing

CREATE TABLE analytics.events_design
(
    event_time DateTime,
    country    LowCardinality(String),
    user_id    UInt64,
    event_type LowCardinality(String)
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(event_time)
ORDER BY (country, event_time, user_id)
SETTINGS index_granularity = 8192;

EXPLAIN indexes = 1
SELECT count() FROM analytics.events_design
WHERE country = 'DE' AND event_time >= '2026-10-01';
```

Then confirm the choice against production traffic in `system.query_log`, and revisit it when the query mix changes.

Common mistake: leading with the most selective column. If `user_id` is nearly unique, `ORDER BY (user_id, event_time)` looks attractive, but a query that filters only time cannot use the index and scans every granule. Matching the common filter prefix beats raw selectivity. If two sort orders are both common, keep one as `ORDER BY` and add the other as a projection.

## How do you batch inserts and keep the number of parts low?

Part count is the main write-side health metric. Every insert creates a part per partition it touches. Merges combine parts, but a merge costs far more CPU per byte than an insert, so the insert rate must stay under the merge throughput. The cheapest control is to send fewer, larger inserts.

```sql
-- Parts per partition right now
SELECT table, partition, count() AS parts, sum(rows) AS rows
FROM system.parts
WHERE active AND database = 'analytics'
GROUP BY table, partition
ORDER BY parts DESC
LIMIT 20;

-- Thresholds that first delay, then reject inserts
SELECT name, value, changed, description
FROM system.merge_tree_settings
WHERE name IN ('parts_to_delay_insert', 'parts_to_throw_insert', 'max_parts_in_total');
```

Target 10k to 100k rows per insert on a busy table, and let async inserts buffer many small writers into one part.

Common mistake: one `INSERT` per event from an application. Each call becomes a part, merges fall behind, and `parts_to_delay_insert` starts throttling the writer. Buffer in the client, or set `async_insert = 1` with `wait_for_async_insert = 1` for clients that cannot batch. Do not raise the delay and throw thresholds to work around the backlog; fix the insert pattern instead.

## How do you choose codecs, LowCardinality, and skipping structures?

Storage choices compound: a codec shrinks a column, `LowCardinality` shrinks a string column's keys, and a projection or skip index trades storage for fewer scanned granules. Choose each one from the column's value pattern, then measure the real ratio in `system.columns`.

```sql
CREATE TABLE analytics.metrics_tuned
(
    ts        DateTime CODEC(DoubleDelta, ZSTD(1)),
    device_id UInt64   CODEC(Delta, ZSTD(1)),
    status    LowCardinality(String),
    value     Float64  CODEC(Gorilla, ZSTD(1)),
    payload   String   CODEC(ZSTD(3))
)
ENGINE = MergeTree()
ORDER BY (device_id, ts);

-- Measure compression per column
SELECT name,
       formatReadableSize(sum(data_compressed_bytes)) AS compressed,
       round(sum(data_uncompressed_bytes) / sum(data_compressed_bytes), 2) AS ratio
FROM system.columns
WHERE table = 'metrics_tuned'
GROUP BY name
ORDER BY sum(data_compressed_bytes) DESC;
```

Use `Delta` or `DoubleDelta` for timestamps and counters, `T64` for small integer ranges, `Gorilla` for slowly changing floats, and `ZSTD` as the general backstop.

Common mistake: selecting `LowCardinality` for a column that is actually unique. The dictionary grows to the size of the data and every read pays a lookup. Check `uniqExact` first. The same rule applies to skipping structures: add a skip index or projection only after `system.query_log` proves a query scans too many marks, because both cost storage and merge time.

## How do you tune JOINs and distributed query fan-out?

A join builds a hash table from the right table, so the right side must be the small one. Pick the algorithm by whether that table fits in memory: `hash` when it does, `grace_hash` to spill when it does not, `direct` when the right table is tiny and keyed.

For distributed queries, the initiator sends the query to every shard and merges the results. Latency is close to the slowest shard, and network traffic grows with the number of shards times the partial result each one returns. A shard key of `rand()` spreads rows evenly; a low-cardinality shard key creates a hot shard.

```sql
SET join_algorithm = 'grace_hash';
SET max_bytes_in_join = 20000000000;

SELECT c.region, count() AS events
FROM analytics.events AS e
LEFT JOIN analytics.country_ref AS c ON e.country = c.country
GROUP BY c.region
ORDER BY events DESC;

-- Fan-out cost seen from the initiator
SELECT query, read_rows, result_rows, memory_usage
FROM system.query_log
WHERE type = 'QueryFinish' AND query LIKE '%JOIN%'
ORDER BY read_rows DESC
LIMIT 5;
```

Keep lookup tables in dictionaries when the same join repeats, because a dictionary lookup is a hash call instead of a join.

Common mistake: joining two `Distributed` tables without `GLOBAL`. Each shard then builds a hash table from only its own slice of the right table, so rows on different shards never match. Use `GLOBAL JOIN`, or set `distributed_product_mode = 'global'` when cross-shard joins are unavoidable. Prefer denormalizing the dimension into the fact table instead.

## Real world use cases for ClickHouse

These patterns recur in production. Each one leans on a different part of the engine: incremental materialized views for freshness, TTL and storage policies for retention, `ReplacingMergeTree` for mutable entities, and table functions for reading object storage in place.

### 1. Real-time product analytics

A consumer app streams click and purchase events through Kafka. A materialized view keeps per-minute rollups, and dashboards query the rollup table with sub-second latency while raw events stay for three days.

```sql
SELECT toStartOfMinute(event_time) AS minute,
       country,
       uniqMerge(users) AS users,
       sum(revenue)     AS revenue
FROM analytics.events_minute
WHERE minute >= now() - INTERVAL '6' HOUR
  AND country = 'DE'
GROUP BY minute
ORDER BY minute;
```

Why ClickHouse: incremental materialized views mean the dashboard never scans raw events, and the columnar engine aggregates millions of rows per query without a pre-computation job.

### 2. Observability and log analytics with retention tiers

A platform stores structured application logs, keeps 7 days on fast SSD and 90 days on cheaper storage, and searches messages with a token index.

```sql
CREATE TABLE observability.logs
(
    log_time DateTime,
    service  LowCardinality(String),
    level    LowCardinality(String),
    message  String CODEC(ZSTD(3)),
    INDEX idx_message message TYPE tokenbf_v1(8192, 3, 0) GRANULARITY 4
)
ENGINE = MergeTree()
PARTITION BY toDate(log_time)
ORDER BY (service, level, log_time)
TTL log_time + INTERVAL 7 DAY TO VOLUME 'hot',
    log_time + INTERVAL 90 DAY DELETE;
```

Why ClickHouse: TTL storage policies implement retention without a batch job, and skipping indexes keep free-text queries inside interactive latency.

### 3. CDC mirror of an operational database

Debezium events land in a `ReplacingMergeTree` keyed by primary key. A periodic `OPTIMIZE` keeps the table compact, and readers that need a fully deduplicated view use `FINAL` or `argMax` as described above.

```sql
CREATE TABLE cdc.customers_rt
(
    customer_id UInt64,
    name        String,
    email       String,
    updated_at  DateTime,
    is_deleted  UInt8 DEFAULT 0
)
ENGINE = ReplacingMergeTree(updated_at, is_deleted)
ORDER BY customer_id;
```

Why ClickHouse: `ReplacingMergeTree` gives last-write-wins semantics without a merge-on-read table format, and the columnar layout still answers analytical questions over the mirrored data.

### 4. Querying a lakehouse in place

A data team keeps Parquet and Iceberg tables in S3. Instead of loading everything, they query the lake directly with the `iceberg` and `s3` table functions from the same cluster that serves dashboards.

```sql
SELECT country, count() AS events, sum(amount) AS revenue
FROM iceberg('s3://acme-warehouse/analytics/events/', 'access_key', 'secret_key')
WHERE event_date = '2026-10-04'
GROUP BY country
ORDER BY revenue DESC;
```

Why ClickHouse: the same engine that serves real-time dashboards can scan object storage without a separate ingestion pipeline, which shortens the path from raw files to an answer.

### 5. Multi-tenant usage metering and billing

A SaaS platform meters API calls per customer per hour, sums counters with `SummingMergeTree`, and reconciles invoices with the same rollup table.

```sql
SELECT tenant_id,
       sum(calls) AS calls,
       round(sum(errors) / nullIf(sum(calls), 0), 4) AS error_rate
FROM billing.usage_hourly
WHERE hour >= toStartOfMonth(now())
GROUP BY tenant_id
ORDER BY calls DESC;
```

Why ClickHouse: `SummingMergeTree` keeps additive counters compact and merge-friendly, and the same rollup serves both live usage dashboards and month-end reconciliation.

## ClickHouse exercises

These exercises use only the Python standard library and model ClickHouse layout and tuning arithmetic. Each one names the task, gives a hint, states the expected result, then prints and asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers are estimates, not promises. Real results depend on types, cardinality, sort order, codec, and cluster shape; the value of each exercise is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Choose an ORDER BY key from query patterns

**Task.** Three of four query patterns filter on `country` and `event_time`. Choose the leading `ORDER BY` columns from the columns common to all patterns, ordered by cardinality.

**Hint.** Intersect the `where` lists, then sort the shared columns by cardinality so the lowest-cardinality column leads.

**Expected result.** `("country", "event_time")`.

In [ ]:
QUERY_PATTERNS = [
    {"where": ["country", "event_time"]},
    {"where": ["country", "event_time", "user_id"]},
    {"where": ["country", "event_time"]},
]
CARDINALITY = {"country": 200, "event_time": 500_000_000, "user_id": 50_000_000}

def choose_order_by(patterns, cardinality):
    common = set(patterns[0]["where"])
    for pattern in patterns[1:]:
        common &= set(pattern["where"])
    return tuple(sorted(common, key=lambda column: cardinality[column]))

key = choose_order_by(QUERY_PATTERNS, CARDINALITY)
print("ORDER BY", key)
print("leading column appears in every pattern:", all("country" in p["where"] for p in QUERY_PATTERNS))

assert key == ("country", "event_time")
assert key[0] == "country"

### Try it 2. Compute granule count from rows and index_granularity

**Task.** For 5 billion rows and `index_granularity = 8192`, compute the number of granules and the number of marks if the primary index covers three key columns.

**Hint.** `granules = ceil(rows / index_granularity)`; marks are one per granule per indexed column.

**Expected result.** 610,352 granules and 1,831,056 marks.

In [ ]:
import math

ROWS = 5_000_000_000
INDEX_GRANULARITY = 8192
KEY_COLUMNS = 3

granules = math.ceil(ROWS / INDEX_GRANULARITY)
marks = granules * KEY_COLUMNS
print(f"granules: {granules:,}")
print(f"marks for {KEY_COLUMNS} key columns: {marks:,}")

assert granules == 610_352
assert marks == 1_831_056

### Try it 3. Decide skip index versus projection

**Task.** Return the right structure for each workload: a bounded filter column, a second frequent sort order, or a high-cardinality column that is never filtered alone.

**Hint.** A second sort order needs a projection; a bounded filter column needs a skip index; otherwise add nothing.

**Expected result.** `skip index`, `projection`, `no extra index`.

In [ ]:
MAX_SKIP_INDEX_DISTINCT = 10_000

def choose_structure(distinct_values, filtered_alone, needs_second_sort_order):
    if needs_second_sort_order:
        return "projection"
    if filtered_alone and distinct_values <= MAX_SKIP_INDEX_DISTINCT:
        return "skip index"
    return "no extra index"

cases = [
    (500, True, False),
    (50_000_000, True, True),
    (50_000_000, False, False),
]
for case in cases:
    print(case, "->", choose_structure(*case))

assert choose_structure(500, True, False) == "skip index"
assert choose_structure(50_000_000, True, True) == "projection"
assert choose_structure(50_000_000, False, False) == "no extra index"

### Try it 4. Estimate parts growth from insert cadence

**Task.** A writer sends 120 single-row inserts per minute. Merge capacity is 1.5 parts per second. Compute parts per day, merge capacity per day, and parts per day if the writer batches 1,000 inserts into one.

**Hint.** `parts_per_day = inserts_per_minute * 1440`; batching divides it by the batch factor.

**Expected result.** 172,800 parts, 129,600 merges of capacity, 173 parts when batched.

In [ ]:
import math

INSERTS_PER_MINUTE = 120
MERGE_PARTS_PER_SECOND = 1.5
BATCH_FACTOR = 1_000
MINUTES_PER_DAY = 24 * 60

parts_per_day = INSERTS_PER_MINUTE * MINUTES_PER_DAY
merge_capacity_per_day = int(MERGE_PARTS_PER_SECOND * 86_400)
parts_if_batched = math.ceil(parts_per_day / BATCH_FACTOR)

print(f"parts per day: {parts_per_day:,}")
print(f"merge capacity per day: {merge_capacity_per_day:,}")
print(f"parts per day when batched: {parts_if_batched:,}")

assert parts_per_day == 172_800
assert parts_per_day > merge_capacity_per_day
assert parts_if_batched == 173

### Try it 5. Pick a codec by column type

**Task.** Choose a codec for a timestamp, a float gauge, and an already-compressed blob.

**Hint.** Delta-family codecs for ordered values, `Gorilla` for slowly changing floats, `NONE` for data that is already compressed.

**Expected result.** `DoubleDelta, ZSTD(1)`, `Gorilla, ZSTD(1)`, `NONE`.

In [ ]:
def choose_codec(column_type, monotonic, is_float, already_compressed):
    if already_compressed:
        return "NONE"
    if column_type == "DateTime" or monotonic:
        return "DoubleDelta, ZSTD(1)"
    if is_float:
        return "Gorilla, ZSTD(1)"
    return "ZSTD(1)"

for args in [
    ("DateTime", True, False, False),
    ("Float64", False, True, False),
    ("String", False, False, True),
]:
    print(args, "->", choose_codec(*args))

assert choose_codec("DateTime", True, False, False) == "DoubleDelta, ZSTD(1)"
assert choose_codec("Float64", False, True, False) == "Gorilla, ZSTD(1)"
assert choose_codec("String", False, False, True) == "NONE"

### Try it 6. Compute compression ratio from read_bytes

**Task.** A table holds 1,200 GiB uncompressed and 150 GiB on disk. Compute the compression ratio and the space saved.

**Hint.** `ratio = uncompressed / compressed`; saved is the difference.

**Expected result.** Ratio 8.0, saving 1,050 GiB.

In [ ]:
UNCOMPRESSED_GIB = 1_200
COMPRESSED_GIB = 150

ratio = UNCOMPRESSED_GIB / COMPRESSED_GIB
saved_gib = UNCOMPRESSED_GIB - COMPRESSED_GIB
print(f"ratio: {ratio:.1f}x")
print(f"saved: {saved_gib:,} GiB")

assert ratio == 8.0
assert saved_gib == 1_050

### Try it 7. Model distributed fan-out cost

**Task.** A single node scans a table in 2.0 seconds. Model an 8-shard cluster that splits the rows evenly, then compute the partial rows returned to the initiator when each shard returns 10,000 groups.

**Hint.** Each shard scans `rows / shards`; the initiator receives `shards * partial_rows` and re-aggregates them.

**Expected result.** 500,000,000 rows per shard, 0.25 s ideal scan time, 80,000 partial rows at the initiator.

In [ ]:
SHARDS = 8
ROWS_TOTAL = 4_000_000_000
SINGLE_NODE_SECONDS = 2.0
PARTIAL_GROUPS_PER_SHARD = 10_000

rows_per_shard = ROWS_TOTAL / SHARDS
ideal_scan_seconds = SINGLE_NODE_SECONDS / SHARDS
rows_to_initiator = SHARDS * PARTIAL_GROUPS_PER_SHARD

print(f"rows per shard: {rows_per_shard:,.0f}")
print(f"ideal scan time: {ideal_scan_seconds:.2f} s")
print(f"partial rows at initiator: {rows_to_initiator:,}")

assert rows_per_shard == 500_000_000
assert ideal_scan_seconds == 0.25
assert rows_to_initiator == 80_000

### Try it 8. Check whether a hash join fits in memory

**Task.** The right table has 20 million rows at 120 bytes each. The memory limit is 2 GiB. Decide between `hash` and `grace_hash`.

**Hint.** Build size is `rows * bytes_per_row`; compare it with the limit.

**Expected result.** 2.4 GB does not fit under 2 GiB, so choose `grace_hash`.

In [ ]:
RIGHT_ROWS = 20_000_000
BYTES_PER_ROW = 120
MAX_BYTES_IN_JOIN = 2 * 1024**3

build_bytes = RIGHT_ROWS * BYTES_PER_ROW
fits = build_bytes <= MAX_BYTES_IN_JOIN
algorithm = "hash" if fits else "grace_hash"

print(f"build side: {build_bytes / 1e9:.1f} GB")
print(f"limit:      {MAX_BYTES_IN_JOIN / 1e9:.2f} GB")
print("algorithm:", algorithm)
print("better fix: put the smaller table on the right")

assert build_bytes == 2_400_000_000
assert not fits
assert algorithm == "grace_hash"